# Notebook setup

In [ ]:
from pathlib import Path
import os

# show where the notebook is running
print("CWD before:", Path.cwd())

# Point to your package (adjust if needed)
# e.g. if your modules are under src/, add it to sys.path
import sys
sys.path.append(str(Path.cwd()))  # or Path("src").resolve()

print("CWD after:", Path.cwd())
# from data_processing.logging_utils import logger
# from data_processing.data_setup import create_data_directory

In [ ]:
import pandas as pd
import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np

# from data_processing.integrated_data_preprocessor import IntegratedICUPreprocessor
# from cohort_data import get_cohort_hadm_ids_and_targets
# from logging_utils import logger
# from data_processing.data_setupd import create_data_directory

# Input CSV file paths
INITIAL_COHORT_CSV = "../csvs/initial_cohort.csv"    # Training/validation patient IDs
TEST_EXAMPLE_CSV = "../csvs/test_example.csv"        # Test set patient IDs

# Output directory for processed data
DATA_DIR = "data"

df_init = pd.read_csv(INITIAL_COHORT_CSV)
df_test = pd.read_csv(TEST_EXAMPLE_CSV)
display(df_init.head()); display(df_test.head())
print("init shape:", df_init.shape, "test shape:", df_test.shape)

# For faster debug runs, sample a small subset (e.g., 100 patients)
INIT_SAMPLE_N = 100
TEST_SAMPLE_N = 50

init_ids = df_init["subject_id"].astype(int).sample(min(INIT_SAMPLE_N, len(df_init)), random_state=42).tolist()
test_ids = df_test["subject_id"].astype(int).sample(min(TEST_SAMPLE_N, len(df_test)), random_state=42).tolist()

print(len(init_ids), len(test_ids))


# Output file names for saved artifacts
PREPROCESSOR_FILE = "integrated_preprocessor.pkl"  # Fitted preprocessing pipeline
TRAIN_DATA_FILE = "train_data.pkl"                 # Training dataset
VAL_DATA_FILE = "val_data.pkl"                     # Validation dataset  
TEST_DATA_FILE = "test_data.pkl"                   # Test dataset

## DB access sanity + cohort/targets only

In [ ]:
import duckdb
from data_processing.data_extraction import get_cohort_hadm_ids_and_targets, DUCKDB_PATH  # or set your own path here

# db = Path("/mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb")
    # assert db.exists(), f"DB not found at {db}"
print("DUCKDB_PATH ->", DUCKDB_PATH)
# con = duckdb.connect(DUCKDB_PATH)
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# con = duckdb.connect(str(db), read_only=True)

# sanity checks
print(con.execute("PRAGMA database_list").fetchdf())
print(con.execute("SHOW TABLES").fetchdf().head())


hadm_ids, targets = get_cohort_hadm_ids_and_targets(con, init_ids)
print("cohort #hadm:", len(hadm_ids))
print("targets shape:", targets.shape)
pd.DataFrame(targets, columns=["mortality","los>7d","readm≤30d"]).head()

In [ ]:
# label distribution check (stratification sanity)
lab = pd.DataFrame(targets, columns=["mortality","los","readm"])
lab["combo"] = (lab["mortality"].astype(int).astype(str) +
                lab["los"].astype(int).astype(str) +
                lab["readm"].astype(int).astype(str))
lab["combo"].value_counts(normalize=True).rename("freq").to_frame()

## Static features only

In [ ]:
from data_processing.static_data import get_static_data, STATIC_COLUMNS

static_raw = get_static_data(con, hadm_ids)
print("static_raw shape:", static_raw.shape)

# peek as DataFrame with original column ordering
# from yourpkg.static_data import STATIC_COLUMNS
pd.DataFrame(static_raw, columns=STATIC_COLUMNS).head()

## Time-series features only

In [ ]:
from data_processing.timeseries_data import get_timeseries_data, TIMESERIES_COLUMNS

ts_data, ts_miss = get_timeseries_data(con, hadm_ids)
# con.close()

print("ts_data:", ts_data.shape, "ts_miss:", ts_miss.shape)
# tiny peek
N, H, F = ts_data.shape
ts_data[0, :3, :8]  # first patient, first 3 hours, first 8 features
pd.DataFrame(ts_data, columns=TIMESERIES_COLUMNS).head()